In [1]:
import sqlite3
import pandas as pd
import numpy as np

db_path = "data/data.db"

conn = sqlite3.connect(db_path)


In [2]:
df = pd.read_sql_query("""
    SELECT *
    FROM ohlc_hw
""", conn,parse_dates=['ts'])

display(df.info(show_counts=True))
display(df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 397711 entries, 0 to 397710
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   ts         397711 non-null  datetime64[ns]
 1   open       397711 non-null  float64       
 2   high       397711 non-null  float64       
 3   low        397711 non-null  float64       
 4   close      397711 non-null  float64       
 5   adj_close  377856 non-null  float64       
 6   volume     397711 non-null  int64         
 7   ticker     397711 non-null  object        
dtypes: datetime64[ns](1), float64(5), int64(1), object(1)
memory usage: 24.3+ MB


None

,ts,open,high,low,close,adj_close,volume
count,397711,397711.000000,397711.000000,397711.000000,397711.000000,377856.000000,3.977110e+05
mean,1998-04-17 21:18:45.292485888,46.285479,46.747660,45.816279,46.292903,51.652250,3.097773e+07
min,1962-01-02 00:00:00,0.000000,0.005208,0.004801,0.005208,0.002945,0.000000e+00
25%,1985-05-28 00:00:00,4.209938,4.240000,4.160000,4.200000,1.390000,1.730700e+06
50%,2000-01-06 00:00:00,19.850000,20.060000,19.630000,19.860000,11.960000,4.648200e+06
75%,2012-08-21 00:00:00,57.530000,58.130000,56.950000,57.530000,43.770000,1.102380e+07
max,2024-12-06 00:00:00,604.260000,608.630000,597.880000,605.400000,10442.510783,9.276606e+09
std,NaN,67.132594,67.778210,66.477178,67.141879,194.584045,1.266237e+08


### 1.2 Data quality checks

Use a copy to normalize ticker labels before grouping. Keep the original data
and labels so we can compare them after repairs.


In [3]:
raw_df = df.copy()

def prepare_diagnosis(data):
    out = data.copy().reset_index(drop=True)
    if "ticker_original" not in out:
        out["ticker_original"] = out["ticker"]
    ticker = out["ticker"].astype("string")
    out["ticker"] = (
        out["ticker"].astype("string")
        .str.replace(r"<[^>]*>", "", regex=True)
        .str.strip().str.upper().replace("", pd.NA)
    )
    out["bad_ticker_format"] = ticker.ne(out["ticker"]).fillna(False)
    out["ts"] = pd.to_datetime(out["ts"], errors="coerce").dt.normalize()
    return out

display(df.isna().sum())


ts               0
open             0
high             0
low              0
close            0
adj_close    19855
volume           0
ticker           0
dtype: int64

In [4]:
ref = pd.read_sql_query(
    "SELECT ticker, ts FROM ohlc", conn, parse_dates=["ts"]
)

def missing_trading_days(data, ref, verbose=True):
    data = prepare_diagnosis(data)
    ref = prepare_diagnosis(ref)
    gaps = {}

    for ticker, history in ref.groupby("ticker"):
        expected = pd.DatetimeIndex(history["ts"].dropna()).unique().sort_values()
        actual = pd.DatetimeIndex(
            data.loc[data["ticker"] == ticker, "ts"].dropna()
        ).unique()
        gaps[ticker] = expected.difference(actual)

        if verbose:
            print(
                f"Ticker {ticker} is missing {len(gaps[ticker])} trading days "
                f"between {expected.min().date()} and {expected.max().date()}"
            )

    return gaps

gaps = missing_trading_days(df, ref)


Ticker AAPL is missing 0 trading days between 1980-12-12 and 2024-10-18
Ticker AMGN is missing 0 trading days between 1983-06-17 and 2024-10-18
Ticker AMZN is missing 0 trading days between 1997-05-15 and 2024-12-06
Ticker AXP is missing 0 trading days between 1972-04-03 and 2024-10-18
Ticker BA is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker CAT is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker CRM is missing 0 trading days between 2004-06-23 and 2024-10-18
Ticker CSCO is missing 0 trading days between 1990-02-16 and 2024-10-18
Ticker CVX is missing 0 trading days between 1962-01-02 and 2024-12-06
Ticker DIS is missing 0 trading days between 1962-01-02 and 2024-12-06
Ticker GS is missing 0 trading days between 1999-05-04 and 2024-10-18
Ticker HD is missing 0 trading days between 1981-09-22 and 2024-10-18
Ticker HON is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker IBM is missing 0 trading days between 1962-01-02 and 2024-12-06


Ticker JNJ is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker JPM is missing 0 trading days between 1969-03-05 and 2024-10-18
Ticker KO is missing 0 trading days between 1962-01-02 and 2024-12-06
Ticker MCD is missing 0 trading days between 1966-07-05 and 2024-10-18
Ticker MMM is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker MRK is missing 0 trading days between 1962-01-02 and 2024-12-06
Ticker MSFT is missing 0 trading days between 1986-03-13 and 2024-10-18
Ticker NKE is missing 0 trading days between 1980-12-02 and 2024-12-06
Ticker NVDA is missing 0 trading days between 1999-01-22 and 2024-10-18
Ticker PG is missing 0 trading days between 1962-01-02 and 2024-12-06
Ticker SHW is missing 0 trading days between 1968-01-02 and 2024-10-18
Ticker TRV is missing 0 trading days between 1972-04-03 and 2024-10-18
Ticker UNH is missing 0 trading days between 1984-10-17 and 2024-10-18


Ticker V is missing 0 trading days between 2008-03-19 and 2024-10-18
Ticker VZ is missing 0 trading days between 1983-11-21 and 2024-10-18
Ticker WMT is missing 0 trading days between 1972-04-03 and 2024-10-18


In [5]:

def flag_outliers(df):
    out = prepare_diagnosis(df).sort_values(["ticker", "ts"]).copy()
    original_out = out.copy()
    business_cols = ["ticker", "ts", "open", "high", "low", "close",
                     "adj_close", "volume"]
    out = out.drop_duplicates(business_cols).copy()
    conflicting = out.duplicated(["ticker", "ts"], keep=False)

    price_cols = ["open", "high", "low", "close", "adj_close"]
    prices = out[price_cols].apply(pd.to_numeric, errors="coerce")
    bad_prices = prices.le(0) | ~np.isfinite(prices)

    for col in price_cols:
        out[f"bad_{col}"] = bad_prices[col]

    out["bad_price"] = bad_prices.any(axis=1)

    out["bad_ohlc_bounds"] = (
        prices["high"].lt(prices["close"])
        | prices["low"].gt(prices["close"])
    )

    close = prices["close"].where(~bad_prices["close"] & ~conflicting)
    prev_close = close.groupby(out["ticker"]).shift(1)

    out["daily_return"] = (
        (close - prev_close) / prev_close
    ).replace([np.inf, -np.inf], np.nan)

    grouped = out.groupby("ticker")["daily_return"]
    out["mean_return"] = grouped.transform("mean")
    out["std_return"] = grouped.transform("std", ddof=1)

    std = out["std_return"]

    out["return_screen_status"] = np.select(
        [std.isna(), std.eq(0)],
        ["undefined_std", "zero_std"],
        default="ok",
    )

    out["return_outlier"] = (
        out["daily_return"].notna()
        & std.gt(0)
        & (
            (out["daily_return"] - out["mean_return"]).abs()
            > 3 * std
        )
    )

    out["review_flag"] = out[
        ["return_outlier", "bad_ohlc_bounds", "bad_price"]
    ].any(axis=1)

    flag_cols = [f"bad_{col}" for col in price_cols] + [
        "bad_price", "bad_ohlc_bounds", "daily_return", "mean_return",
        "std_return", "return_screen_status", "return_outlier", "review_flag"
    ]
    original_out = original_out.drop(columns=flag_cols, errors="ignore")
    return original_out.merge(
        out[business_cols + flag_cols], on=business_cols,
        how="left", validate="many_to_one"
    )
checked = flag_outliers(df)
display(checked.head())

,ts,open,high,low,close,adj_close,volume,ticker,ticker_original,bad_ticker_format,...,bad_close,bad_adj_close,bad_price,bad_ohlc_bounds,daily_return,mean_return,std_return,return_screen_status,return_outlier,review_flag
0,1980-12-12,0.12835,0.12835,0.12835,0.12835,0.098941,469034068,AAPL,AAPL,False,...,False,False,False,False,NaN,0.001074,0.027846,ok,False,False
1,1980-12-15,0.12165,0.12165,0.12165,0.12165,0.093779,175884971,AAPL,AAPL,False,...,False,False,False,False,-0.052201,0.001074,0.027846,ok,False,False
2,1980-12-16,0.11272,0.11272,0.11272,0.11272,0.086896,105728103,AAPL,AAPL,False,...,False,False,False,False,-0.073407,0.001074,0.027846,ok,False,False
3,1980-12-17,0.11551,0.11551,0.11551,0.11551,0.089046,86441684,AAPL,AAPL,False,...,False,False,False,False,0.024752,0.001074,0.027846,ok,False,False
4,1980-12-18,0.11886,0.11886,0.11886,0.11886,0.091627,73449671,AAPL,AAPL,False,...,False,False,False,False,0.029002,0.001074,0.027846,ok,False,False


In [6]:
%who

checked	 conn	 db_path	 df	 flag_outliers	 gaps	 missing_trading_days	 np	 pd	 
prepare_diagnosis	 raw_df	 ref	 sqlite3	 


### Data Quality Report

- **High:** unusable prices/volume, impossible bounds, or conflicting records.
- **Medium:** missing dates, missing/zero volume, or returns needing review.
- **Low:** formatting or exact repeated records.


In [7]:
def flag_inconsistencies(df):
    out = df.copy()
    out["bad_open_bounds"] = (
        out["open"].lt(out["low"])
        | out["open"].gt(out["high"])
        | out["low"].gt(out["high"])
    )
    out["bad_volume"] = (
        out["volume"].notna()
        & (~np.isfinite(out["volume"]) | out["volume"].lt(0))
    )
    out["zero_volume"] = out["volume"].eq(0)
    return out


In [8]:
def build_data_quality_report(df, ref):
    out = flag_outliers(df)
    out = flag_inconsistencies(out)
    price_cols = ["open", "high", "low", "close", "adj_close"]
    business_cols = ["ticker", "ts"] + price_cols + ["volume"]
    report_cols = ["ticker", "issue_type", "issue_date", "description", "severity"]

    out["exact_duplicate"] = out.duplicated(business_cols)
    out["conflicting_key"] = (
        out.groupby(["ticker", "ts"])[price_cols + ["volume"]]
        .transform("nunique", dropna=False).gt(1).any(axis=1)
    )

    checks = [
        ("bad_price", out["bad_price"],
         "Missing, nonpositive, or non-finite price, including adjusted close.", "high"),
        ("bad_ohlc_bounds", out["bad_ohlc_bounds"],
         "Raw close is outside low/high.", "high"),
        ("bad_open_bounds", out["bad_open_bounds"],
         "Raw open is outside low/high, or low exceeds high.", "high"),
        ("bad_volume", out["bad_volume"], "Non-finite or negative volume.", "high"),
        ("zero_volume", out["zero_volume"], "Zero volume needs review.", "medium"),
        ("return_outlier", out["return_outlier"],
         "Return differs from its ticker mean by >3 sample std; review only.", "medium"),
        ("exact_duplicate", out["exact_duplicate"], "Exact repeated record.", "low"),
        ("conflicting_key", out["conflicting_key"],
         "Different records share ticker/date; investigate both.", "high"),
        ("bad_ticker_format", out["bad_ticker_format"],
         "Ticker markup, whitespace, or case needs normalization.", "low"),
    ]
    for col in business_cols:
        checks.append(("missing_value", out[col].isna(), f"{col} is missing.",
                       "medium" if col == "volume" else "high"))

    report = []
    for issue, mask, description, severity in checks:
        rows = out.loc[mask, ["ticker", "ts"]].rename(columns={"ts": "issue_date"}).copy()
        rows["issue_type"] = issue
        rows["description"] = description
        rows["severity"] = severity
        report.append(rows[report_cols])

    gaps = missing_trading_days(df, ref, verbose=False)
    for ticker, dates in gaps.items():
        report.append(pd.DataFrame({
            "ticker": ticker, "issue_type": "missing_trading_day", "issue_date": dates,
            "description": "Absent date from this ticker's supplied ohlc history.",
            "severity": "medium",
        })[report_cols])

    return pd.concat(report, ignore_index=True).sort_values(
        ["ticker", "issue_date", "issue_type"]
    ).reset_index(drop=True)


In [9]:
data_quality_report_before = build_data_quality_report(df, ref)
data_quality_report = data_quality_report_before

display(data_quality_report.head(10))
display(data_quality_report.groupby(["issue_type", "severity"]).size().rename("issues"))



C:\Users\weita\AppData\Local\Temp\ipykernel_43668\4189027246.py:51: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(report, ignore_index=True).sort_values(


,ticker,issue_type,issue_date,description,severity
0,AAPL,bad_ticker_format,1980-12-19,"Ticker markup, whitespace, or case needs norma...",low
1,AAPL,return_outlier,1980-12-26,Return differs from its ticker mean by >3 samp...,medium
2,AAPL,bad_ticker_format,1980-12-29,"Ticker markup, whitespace, or case needs norma...",low
3,AAPL,exact_duplicate,1981-01-02,Exact repeated record.,low
4,AAPL,exact_duplicate,1981-01-05,Exact repeated record.,low
5,AAPL,bad_price,1981-01-08,"Missing, nonpositive, or non-finite price, inc...",high
6,AAPL,missing_value,1981-01-08,adj_close is missing.,high
7,AAPL,bad_price,1981-01-13,"Missing, nonpositive, or non-finite price, inc...",high
8,AAPL,missing_value,1981-01-13,adj_close is missing.,high
9,AAPL,bad_ticker_format,1981-02-02,"Ticker markup, whitespace, or case needs norma...",low


issue_type         severity
bad_open_bounds    high            5
bad_price          high        19860
bad_ticker_format  low         39936
exact_duplicate    low         29460
missing_value      high        19855
return_outlier     medium       5467
zero_volume        medium         81
Name: issues, dtype: int64

### 1.3


In [10]:
def clean_data(df, ref):
    out = prepare_diagnosis(df)
    out["source_row"] = out.index
    out["ticker_repaired"] = out["bad_ticker_format"]
    out = flag_inconsistencies(flag_outliers(out))
    out["return_outlier_before"] = out["return_outlier"]
    raw_cols = ["open", "high", "low", "close"]
    business_cols = ["ticker", "ts"] + raw_cols + ["adj_close", "volume"]

    all_missing = out[raw_cols].isna().all(axis=1)
    exact_duplicate = out.duplicated(business_cols)
    removed = out.loc[all_missing | exact_duplicate].copy()
    removed["reason"] = np.where(
        all_missing.loc[removed.index], "All raw prices missing", "Exact duplicate"
    )
    out = out.loc[~(all_missing | exact_duplicate)].copy()

    bad_key = out[["ticker", "ts"]].isna().any(axis=1)
    conflicting = out.duplicated(["ticker", "ts"], keep=False)
    quarantined = out.loc[bad_key | conflicting].copy()
    quarantined["reason"] = np.where(
        bad_key.loc[quarantined.index], "Missing ticker/date", "Conflicting ticker/date"
    )
    out = out.loc[~(bad_key | conflicting)].copy()

    for col in raw_cols:
        invalid = ~np.isfinite(out[col]) | out[col].le(0)
        out[f"{col}_repaired"] = out[col].isna() | invalid
        out[col] = out[col].mask(invalid)

    out = flag_inconsistencies(flag_outliers(out))
    impossible = out["bad_ohlc_bounds"] | out["bad_open_bounds"] | out["bad_volume"]
    rejected = out.loc[impossible].copy()
    rejected["reason"] = "Impossible raw bounds or invalid volume"
    quarantined = pd.concat([quarantined, rejected], ignore_index=True)
    out = out.loc[~impossible].sort_values(["ticker", "ts"]).copy()

    out[raw_cols] = out.groupby("ticker")[raw_cols].ffill()
    out["volume_repaired"] = out["volume"].isna()
    out["volume"] = out["volume"].fillna(0)

    out = flag_inconsistencies(flag_outliers(out))
    bad_raw = (~np.isfinite(out[raw_cols]) | out[raw_cols].le(0)).any(axis=1)
    unresolved = bad_raw | out["bad_ohlc_bounds"] | out["bad_open_bounds"]
    rejected = out.loc[unresolved].copy()
    rejected["reason"] = "Leading price gap or invalid bounds after forward-fill"
    quarantined = pd.concat([quarantined, rejected], ignore_index=True)
    out = out.loc[~unresolved].copy()

    ref = prepare_diagnosis(ref)
    assert not ref.duplicated(["ticker", "ts"]).any()
    valid_factor = (
        np.isfinite(ref["close"]) & ref["close"].gt(0)
        & np.isfinite(ref["adj_close"]) & ref["adj_close"].gt(0)
    )
    ref["adjustment_factor"] = (
        ref["adj_close"].where(valid_factor) / ref["close"].where(valid_factor)
    )
    ref = ref.rename(columns={"close": "reference_close"})
    out = out.merge(
        ref[["ticker", "ts", "reference_close", "adjustment_factor"]],
        on=["ticker", "ts"], how="left", validate="many_to_one"
    )

    expected_adj = out["close"] * out["adjustment_factor"]
    factor_available = np.isfinite(out["adjustment_factor"]) & out["adjustment_factor"].gt(0)
    mismatch = factor_available & ~np.isclose(out["adj_close"], expected_adj)
    out["adj_close_repaired"] = (
        ~np.isfinite(out["adj_close"]) | out["adj_close"].le(0)
        | out["close_repaired"] | mismatch
    )
    can_rebuild = factor_available & np.isfinite(expected_adj) & expected_adj.gt(0)
    unresolved = out["adj_close_repaired"] & ~can_rebuild
    rejected = out.loc[unresolved].copy()
    rejected["reason"] = "No verified date-specific adjustment factor"
    quarantined = pd.concat([quarantined, rejected], ignore_index=True)
    out = out.loc[~unresolved].copy()
    repaired = out["adj_close_repaired"]
    out.loc[repaired, "adj_close"] = (
        out.loc[repaired, "close"] * out.loc[repaired, "adjustment_factor"]
    )
    out["adjustment_source"] = np.where(
        repaired, "ohlc: matched ticker/date adj_close / close", "Original adjusted close"
    )

    out = flag_inconsistencies(flag_outliers(out))
    out["close_matches_reference"] = np.isclose(out["close"], out["reference_close"])
    out["return_review"] = np.where(
        out["return_outlier"],
        np.where(out["close_matches_reference"],
                 "Close matches ohlc; event cause unverified",
                 "Repaired/different close; event cause unverified"),
        ""
    )
    return out, removed, quarantined


In [11]:
price_ref = pd.read_sql_query(
    "SELECT ticker, ts, close, adj_close FROM ohlc", conn, parse_dates=["ts"]
)
cleaned_df, removed_df, quarantined_df = clean_data(df, price_ref)

row_counts = pd.Series({
    "input": len(df),
    "retained": len(cleaned_df),
    "removed": len(removed_df),
    "quarantined": len(quarantined_df),
})
display(row_counts.to_frame("rows"))
display(removed_df.groupby("reason").size().rename("rows"))
display(quarantined_df.groupby("reason").size().rename("rows"))

accounted = pd.concat([
    cleaned_df["source_row"], removed_df["source_row"], quarantined_df["source_row"]
])
assert len(accounted) == len(df)
assert set(accounted) == set(range(len(df)))
assert accounted.is_unique
pd.testing.assert_frame_equal(df, raw_df)


,rows
input,397711
retained,368246
removed,29460
quarantined,5


reason
Exact duplicate    29460
Name: rows, dtype: int64

reason
Leading price gap or invalid bounds after forward-fill    5
Name: rows, dtype: int64

### Validate and compare

Retained prices must be positive and finite, bounds valid, keys unique, and
volume finite and nonnegative. Recheck adjusted prices against matched factors.
The after-report can retain medium review flags and missing dates caused by
quarantine; no prices are invented for absent sessions.


In [12]:
def validate_data(df):
    out = flag_inconsistencies(flag_outliers(df))
    price_cols = ["open", "high", "low", "close", "adj_close"]
    assert out[["ticker", "ts"]].notna().all().all()
    assert not out.duplicated(["ticker", "ts"]).any()
    assert np.isfinite(out[price_cols]).all().all()
    assert out[price_cols].gt(0).all().all()
    assert not out["bad_ohlc_bounds"].any()
    assert not out["bad_open_bounds"].any()
    assert np.isfinite(out["volume"]).all() and out["volume"].ge(0).all()
    matched = np.isfinite(out["adjustment_factor"]) & out["adjustment_factor"].gt(0)
    assert np.isclose(
        out.loc[matched, "adj_close"],
        out.loc[matched, "close"] * out.loc[matched, "adjustment_factor"]
    ).all()
    assert matched.loc[out["adj_close_repaired"].eq(True)].all()
    return True

assert validate_data(cleaned_df)
data_quality_report_after = build_data_quality_report(cleaned_df, ref)
assert not data_quality_report_after["severity"].eq("high").any()

display(pd.concat({
    "before": data_quality_report_before.groupby("issue_type").size(),
    "after": data_quality_report_after.groupby("issue_type").size(),
}, axis=1).fillna(0).astype(int))

price_cols = ["open", "high", "low", "close", "adj_close", "volume"]
display(pd.concat({
    "before": df[price_cols].describe(),
    "after": cleaned_df[price_cols].describe(),
}, axis=1))

repair_cols = ["ticker_repaired", "open_repaired", "high_repaired", "low_repaired",
               "close_repaired", "adj_close_repaired", "volume_repaired"]
display(cleaned_df[repair_cols].sum().rename("repaired_rows").to_frame())
display(cleaned_df.loc[
    cleaned_df["return_outlier"],
    ["ticker", "ts", "daily_return", "close", "reference_close", "return_review"]
].head(10))


,before,after
issue_type,,
bad_open_bounds,5,0
bad_price,19860,0
bad_ticker_format,39936,0
exact_duplicate,29460,0
missing_value,19855,0
return_outlier,5467,5057
zero_volume,81,74
missing_trading_day,0,5


before                                               \
                open           high            low          close   
count  397711.000000  397711.000000  397711.000000  397711.000000   
mean       46.285479      46.747660      45.816279      46.292903   
std        67.132594      67.778210      66.477178      67.141879   
min         0.000000       0.005208       0.004801       0.005208   
25%         4.209938       4.240000       4.160000       4.200000   
50%        19.850000      20.060000      19.630000      19.860000   
75%        57.530000      58.130000      56.950000      57.530000   
max       604.260000     608.630000     597.880000     605.400000   

                                            after                 \
           adj_close        volume           open           high   
count  377856.000000  3.977110e+05  368246.000000  368246.000000   
mean       51.652250  3.097773e+07      46.309394      46.771824   
std       194.584045  1.266237e+08      67.143254      67.789319   
min         0.002945  0.000000e+00       0.005208       0.005208   
25%         1.390000  1.730700e+06       4.210000       4.250000   
50%        11.960000  4.648200e+06      19.890000      20.090000   
75%        43.770000  1.102380e+07      57.560000      58.160000   
max     10442.510783  9.276606e+09     604.260000     608.630000   

                                                                  
                 low          close      adj_close        volume  
count  368246.000000  368246.000000  368246.000000  3.682460e+05  
mean       45.840375      46.317325      37.274670  3.095184e+07  
std        66.489450      67.154939      63.587872  1.266153e+08  
min         0.004801       0.005208       0.002945  0.000000e+00  
25%         4.170000       4.210000       1.310000  1.731514e+06  
50%        19.670000      19.910000      11.460000  4.649648e+06  
75%        56.997500      57.570000      41.570000  1.102160e+07  
max       597.880000     605.400000     605.400000  9.276606e+09

,repaired_rows
ticker_repaired,36985
open_repaired,0
high_repaired,0
low_repaired,0
close_repaired,0
adj_close_repaired,28791
volume_repaired,0


,ticker,ts,daily_return,close,reference_close,return_review
9,AAPL,1980-12-26,0.092288,0.158480,0.158480,Close matches ohlc; event cause unverified
76,AAPL,1981-04-02,0.087659,0.117750,0.117750,Close matches ohlc; event cause unverified
197,AAPL,1981-09-25,-0.129776,0.063616,0.063616,Close matches ohlc; event cause unverified
318,AAPL,1982-03-19,0.090173,0.074219,0.074219,Close matches ohlc; event cause unverified
490,AAPL,1982-11-22,-0.089089,0.125560,0.125560,Close matches ohlc; event cause unverified
495,AAPL,1982-11-30,0.103871,0.142300,0.142300,Close matches ohlc; event cause unverified
531,AAPL,1983-01-20,0.111518,0.166850,0.166850,Close matches ohlc; event cause unverified
593,AAPL,1983-04-20,0.088684,0.226000,0.226000,Close matches ohlc; event cause unverified
660,AAPL,1983-07-26,-0.092718,0.174670,0.174670,Close matches ohlc; event cause unverified
686,AAPL,1983-08-31,0.133074,0.166290,0.166290,Close matches ohlc; event cause unverified


In [ ]:
cleaned_to_save = cleaned_df.copy()
cleaned_to_save["ts"] = cleaned_to_save["ts"].dt.strftime("%Y-%m-%d")
cleaned_to_save.to_sql(
    "ohlc_hw_cleaned", conn, if_exists="replace", index=False, chunksize=1000
)
conn.execute(
    "CREATE UNIQUE INDEX ohlc_hw_cleaned_key ON ohlc_hw_cleaned(ticker, ts)"
)
conn.commit()

saved = pd.read_sql_query(
    "SELECT * FROM ohlc_hw_cleaned ORDER BY ticker, ts", conn, parse_dates=["ts"]
)
assert len(saved) == len(cleaned_df)
assert validate_data(saved)
raw_check = pd.read_sql_query("SELECT * FROM ohlc_hw", conn, parse_dates=["ts"])
pd.testing.assert_frame_equal(raw_check, raw_df)


Saved 368,246 validated rows to ohlc_hw_cleaned; ohlc_hw is unchanged.
